# Lending Club Loan pre-issue credit risk model

## Update log

* __20170328__: Initial script
* __20170608__: Updated the code under pandas 0.20.2 version
* __20170610__: Rebuilt core model

## Scope of modeling

This script is used to build the pre-issue credit risk model.

1. We focus on __closed deals__ only and try to classify between __default__ loans and __non-default__ loans. Current ongoing loans and fresh issued loan is not included in the scope of modeling.
2. Since this is a __pre-issue credit risk model__ instead of a model for __real-time default detection__, only __variables collected prior to the issue day of the loan__ will be included as predictor variables (like loan information, current credit history). We do not include variables that are collected real-time throughout the loan period (like last payment day).

In [ ]:
import random
import time

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_curve, auc, make_scorer, confusion_matrix, f1_score

## Load Data from csv

In [ ]:
# read csv (441.8 MB)
loan = pd.read_csv("../data/loan.csv", low_memory=False)
loan_copy = loan.copy()
# check dimension
print loan.shape

In [ ]:
# reset loan variable
loan = loan_copy.copy()

## Set up response variable

In [ ]:
# all categories of loan_status
loan['loan_status'].value_counts()

Loans with status "Current" represent loans that haven't been closed till Jan-2016 (the latest payment month in our data). For this analysis, we focus on closed deals only and try to classify between default and non-default.

The loan_status of closed deals are:

* __Fully Paid__ (non-default)
* __Charged Off__ (default)
* __Default__ (defualt)

In [ ]:
# filter response categories
loan = loan[loan['loan_status'].isin(['Fully Paid','Charged Off','Default'])]
# set up binary response variable
default = loan['loan_status'].map({"Fully Paid": 0.0,"Charged Off": 1.0,"Default": 1.0})
loan['default'] = default
# delete loan_status from data frame
# loan.drop('loan_status', axis=1, inplace=True)

In [ ]:
# check dimension
print loan.shape
print default.shape

## Predictor variables exploration

In [ ]:
# check column names
loan.columns.sort_values()

### 1. Keep pre-issue variables only

Pre-issue variables are variables representing basic loan information, borrower's demographic, income, credit history and other information collected when applying for loan. They are:

* addr_state
* annual_inc
* annual_inc_joint
* application_type
* desc
* earliest_cr_line
* emp_length
* emp_title
* grade
* home_ownership
* id
* initialListStatus
* installment
* int_rate
* issue_d
* loan_amnt
* member_id
* policy_code
* purpose
* pymnt_plan
* sub_grade
* term
* title
* url
* verified_status_joint
* zip_code

In [ ]:
# keep pre-issue variables
loan = loan[['addr_state','annual_inc','annual_inc_joint','application_type','desc','earliest_cr_line',
             'emp_length','emp_title','grade','home_ownership','id','initial_list_status','installment',
             'int_rate','issue_d','loan_amnt','member_id','policy_code','purpose','pymnt_plan',
             'sub_grade','term','title','url','verification_status','zip_code']]
loan.shape

### 2. Drop irrelevant variables.

In [ ]:
loan['title'].str.contains('consolidation|Consolidation').mean()

In [ ]:
loan['emp_title'].str.contains('Bank').mean()

In [ ]:
# drop irrelavent variables
loan.drop(['id','member_id', 'desc', 'url', 'zip_code'], axis=1, inplace=True)
# check dimension
loan.shape

### 2. Check missing values

In [ ]:
# NA count table
NAs = pd.concat([loan.isnull().sum(), np.round(loan.isnull().sum()/len(loan), 2)], axis=1, keys=['missing counts','missing percentage'])

In [ ]:
# print all variables that have NAs
NAs[NAs['missing percentage'] != 0.0].sort_values('missing percentage')

We drop variables that have over 90% NAs. Those variables cannot be used since 

In [ ]:
# drop NA variables
loan.drop(list(NAs[NAs['missing percentage'] > 0.5].index), axis=1, inplace=True)
# check dimension
loan.shape

In [ ]:
loan['mths_since_last_delinq'].hist()

## Data Cleaning & Feature engineering

1. generate new variable using existing variables
2. drop redundant variables
3. change variable to proper data type

__loan_amnt, funded_amnt, funded_amnt_inv__

In [ ]:
# loan_amnt is always greater than funded_amnt and funded_amnt_inv
print (loan['loan_amnt'] >= loan['funded_amnt']).mean()
print (loan['loan_amnt'] >= loan['funded_amnt_inv']).mean()

# add variable perc_funded_inv: The amount committed by investors divided by total loan amount.
loan['perc_funded_amnt'] = loan['funded_amnt'] / loan['loan_amnt']
loan['perc_funded_amnt_inv'] = loan['funded_amnt_inv'] / loan['loan_amnt']

# delete funded_amnt, funded_amnt_inv
loan.drop('funded_amnt', axis=1, inplace=True)
loan.drop('funded_amnt_inv', axis=1, inplace=True)

__grade, sub_grade__

In [ ]:
print loan['grade'].unique()
print loan['sub_grade'].unique()
print len(loan['sub_grade'].unique())

# map the grades to a continuous variable
loan['grade_level'] = loan['sub_grade'].map({'A1':1,'A2':2,'A3':3,'A4':4,'A5':5,
                      'B1':6,'B2':7,'B3':8,'B4':9,'B5':10,
                      'C1':11,'C2':12,'C3':13,'C4':14,'C5':15,
                      'D1':16,'D2':17,'D3':18,'D4':19,'D5':20,
                      'E1':21,'E2':22,'E3':23,'E4':24,'E5':25,
                      'F1':26,'F2':27,'F3':28,'F4':29,'F5':30,
                      'G1':31,'G2':32,'G3':33,'G4':34,'G5':35,})

# delete grade, sub_grade
loan.drop(['grade', 'sub_grade'], axis=1, inplace=True)

__emp_length__: Variable emp_length has 11 categories, we cut the them into 4 different ranges.

In [ ]:
# before cut
print loan['emp_length'].value_counts()

# cut
loan['emp_length'] = loan['emp_length'].map({'10+ years': '10+ years',
                       '9 years': '6-9 years',
                       '8 years': '6-9 years',
                       '7 years': '6-9 years',
                       '6 years': '6-9 years',
                       '5 years': '3-5 years',
                       '4 years': '3-5 years',
                       '3 years': '3-5 years',
                       '2 years': '0-2 years',
                       '1 year': '0-2 years',
                       '< 1 year': '0-2 years',
                       'n/a': '0-2 years' # no employment information
                      })

# after cut
print loan['emp_length'].value_counts()

__pymnt_plan, policy_code, application_type__: Alomost constant. Drop them.

In [ ]:
# drop almost-constant variables
print loan['pymnt_plan'].value_counts()
print loan['policy_code'].value_counts()
print loan['application_type'].value_counts()

# delete them
loan.drop(['pymnt_plan', 'policy_code', 'application_type'], axis=1, inplace=True)

__issue_d, earliest_cr_line__: take the difference to get the number of days since earliest credit line.

In [ ]:
# convert to date
loan['issue_d'] = pd.to_datetime(loan['issue_d'])
loan['earliest_cr_line'] = pd.to_datetime(loan['earliest_cr_line'])

# take the difference
loan['days_since_earliest_cr_line'] = loan['issue_d'] - loan['earliest_cr_line']

# drop issue_d and earliest_cr_line
loan.drop(['issue_d', 'earliest_cr_line'], axis=1, inplace=True)

In [ ]:
loan.columns

__last_pymnt_d, last_credit_pull_d__: take the difference.

In [ ]:
# convert to date
loan['last_pymnt_d'] = pd.to_datetime(loan['last_pymnt_d'])
loan['last_credit_pull_d'] = pd.to_datetime(loan['last_credit_pull_d'])

# take the difference
loan['days_diff_last_pymnt_pull'] = loan['last_credit_pull_d'] - loan['last_pymnt_d']

# drop issue_d and earliest_cr_line
loan.drop(['last_pymnt_d', 'last_credit_pull_d'], axis=1, inplace=True)

Now check data types.

In [ ]:
# NA count table
print loan.shape
NAs = pd.concat([loan.dtypes, loan.isnull().sum(), np.round(loan.isnull().sum()/len(loan), 2)], axis=1, keys=['Type', 'missing counts','missing percentage'])
NAs

In [ ]:
mn.matrix(loan[['mths_since_last_delinq','mths_since_last_major_derog','','tot_cur_bal','tot_coll_amt']])

## Performance Metric

We use AUC (Area-under-the-curve) as an optimization metric. This is a popular metric within credit modeling.

In [ ]:
def get_auc(y_true, y_pred):
    # y: an array of labels marking the actualy result as default (1) or non-default (0) 
    # y_pred: a probability that a row has defaulted
    # fpr: false-positive rate
    # tpr: true-positive rate
    
    fpr, tpr, thresholds = roc_curve(y_true, y_pred)
    auc_score = auc(fpr, tpr)
    return auc_score